<!-- NB_DOC_INTRO_V1 -->
# Flask — API REST CRUD (testable in-process)

> 📚 **Doc thematique** : [docs/07_BDD_DE.md](docs/07_BDD_DE.md) (Bases de donnees / Data Engineering / Web)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Flask** = micro-framework Python pour APIs HTTP. Plus ancien que FastAPI mais toujours present (legacy + ecosystem). Ce notebook execute une **API CRUD complete livres** + tests via `app.test_client()` (sans demarrer un serveur reel).

Pour FastAPI (recommande 2025 pour nouveaux projets) : [FastAPI_API.ipynb](./FastAPI_API.ipynb).

## Plan

1. Setup Flask + test_client
2. Endpoints CRUD (livres)
3. Validation + handlers d'erreur
4. Tests in-process
5. Quand Flask vs FastAPI
6. Deploiement (Gunicorn / Docker)
7. Pieges + References


In [ ]:
import warnings
warnings.filterwarnings("ignore")
try:
    from flask import Flask, jsonify, request, abort
    FLASK_OK = True
    import flask
    print(f"Flask : {flask.__version__}")
except ImportError:
    FLASK_OK = False
    print("Flask pas installe : uv add --group web flask")

## 1. App + endpoints CRUD

In [ ]:
if FLASK_OK:
    app = Flask(__name__)
    app.config["TESTING"] = True

    # Storage in-memory (en prod : Postgres + SQLAlchemy)
    books = [
        {"id": 1, "title": "1984", "author": "Orwell"},
        {"id": 2, "title": "Le Petit Prince", "author": "St-Exupery"},
    ]
    _next_id = 3

    def get_next_id():
        nonlocal_dict = {"v": max((b["id"] for b in books), default=0) + 1}
        return nonlocal_dict["v"]

    @app.route("/books", methods=["GET"])
    def list_books():
        return jsonify(books=books)

    @app.route("/books/<int:book_id>", methods=["GET"])
    def get_book(book_id):
        book = next((b for b in books if b["id"] == book_id), None)
        if not book:
            abort(404, description="Book not found")
        return jsonify(book)

    @app.route("/books", methods=["POST"])
    def create_book():
        if not request.json or "title" not in request.json or "author" not in request.json:
            abort(400, description="Missing title or author")
        new = {"id": get_next_id(), "title": request.json["title"], "author": request.json["author"]}
        books.append(new)
        return jsonify(new), 201

    @app.route("/books/<int:book_id>", methods=["PUT"])
    def update_book(book_id):
        book = next((b for b in books if b["id"] == book_id), None)
        if not book:
            abort(404, description="Book not found")
        if not request.json:
            abort(400, description="No JSON body")
        book["title"] = request.json.get("title", book["title"])
        book["author"] = request.json.get("author", book["author"])
        return jsonify(book)

    @app.route("/books/<int:book_id>", methods=["DELETE"])
    def delete_book(book_id):
        global books
        before = len(books)
        books = [b for b in books if b["id"] != book_id]
        if len(books) == before:
            abort(404, description="Book not found")
        return "", 204

    @app.errorhandler(404)
    def not_found(error):
        return jsonify(error=error.description), 404

    @app.errorhandler(400)
    def bad_request(error):
        return jsonify(error=error.description), 400

    print("App Flask definie avec 5 endpoints CRUD")
else:
    print("SKIP")

## 2. Tests in-process avec test_client (sans serveur)

In [ ]:
if FLASK_OK:
    client = app.test_client()

    # Liste
    r = client.get("/books")
    print(f"GET /books : status={r.status_code}, nb={len(r.json['books'])}")

    # Detail
    r = client.get("/books/1")
    print(f"GET /books/1 : {r.json}")

    # Detail inexistant
    r = client.get("/books/999")
    print(f"GET /books/999 : status={r.status_code} (attendu 404)")

    # Create
    r = client.post("/books", json={"title": "1985", "author": "X"})
    print(f"POST /books : status={r.status_code}, id={r.json['id']}")
    new_id = r.json["id"]

    # Update
    r = client.put(f"/books/{new_id}", json={"title": "1985 v2"})
    print(f"PUT /books/{new_id} : title={r.json['title']}")

    # Delete
    r = client.delete(f"/books/{new_id}")
    print(f"DELETE /books/{new_id} : status={r.status_code}")

    # Verifier delete OK
    r = client.get(f"/books/{new_id}")
    print(f"GET /books/{new_id} : status={r.status_code} (attendu 404)")

    # Bad request
    r = client.post("/books", json={"title": "missing author"})
    print(f"POST malforme : status={r.status_code} (attendu 400)")
else:
    print("SKIP")

## 3. Quand Flask vs FastAPI ?

| Critere | Flask | FastAPI |
|---|---|---|
| Async natif | ❌ | ✅ |
| Validation Pydantic | Manuelle (marshmallow) | Native |
| OpenAPI auto | Non (flasgger en plugin) | Oui gratuit |
| Performance | ~2-3k req/s | ~5-15k req/s |
| Maturite | Enorme (2010+) | Solide (2018+) |
| Pour | Legacy / migration / familier | Nouveau projet |

## 4. Deploiement Flask

### Dev
```bash
flask --app app run --debug
# ou
python app.py
```

### Production (Gunicorn + Nginx)
```bash
pip install gunicorn
gunicorn -w 4 -b 0.0.0.0:8000 app:app
```

### Docker
```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:8000", "app:app"]
```

## 5. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| `app.run(debug=True)` en prod | Werkzeug debugger = RCE — JAMAIS |
| Validation manuelle partout | Marshmallow / pydantic schemas |
| Pas de Blueprint pour gros app | Modulariser via Blueprints |
| `Flask-SQLAlchemy` avec settings hard-codes | `app.config.from_envvar('SETTINGS')` |
| Storage in-memory en prod | DB serieuse (Postgres) |
| Pas de tests | `app.test_client()` est trivial |
| Pas de structured logging | `flask.logging` ou loguru/structlog |


## References

### Documentation
- Flask docs : https://flask.palletsprojects.com/
- Flask testing : https://flask.palletsprojects.com/en/latest/testing/

### Voir aussi
- [FastAPI_API.ipynb](FastAPI_API.ipynb)
- [Structure_BDD_&_DataFrame.ipynb](Structure_BDD_&_DataFrame.ipynb)
- [Streamlit_brique.ipynb](Streamlit_brique.ipynb)
